# Principal Component Analysis (PCA) From Scratch using NumPy

## 🎯 Objective

The objective of this project is to understand how Principal Component Analysis (PCA) works mathematically without using `sklearn.PCA()`.

In this notebook, we will:

- Load the Wine Quality Dataset
- Perform data preprocessing
- Standardize the data
- Calculate the covariance matrix manually
- Compute eigenvalues and eigenvectors
- Select principal components
- Project the data onto a lower-dimensional space
- Visualize the transformed data using Plotly
- Compare Logistic Regression before and after PCA
- Analyze the impact of PCA on accuracy and training time

---

### Dataset

**Wine Quality Dataset**

- Samples : 1599
- Features : 11
- Target : Quality

---

### Technologies Used

- Python
- NumPy
- Pandas
- Plotly
- Scikit-learn

---


# Import Libraries

We first import all the libraries required for data analysis, visualization,
Principal Component Analysis, and Machine Learning.

In [45]:

import numpy as np
import pandas as pd

# Visualization
import plotly.express as px

# Data Preprocessing
from sklearn.preprocessing import StandardScaler

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Measuring Training Time
import time

# Ignore Warning Messages
import warnings
warnings.filterwarnings("ignore")

# Load Dataset

The Wine Quality dataset contains different physicochemical properties of red wine.

Our goal is to reduce the dimensionality of the dataset using PCA while preserving as much information as possible.

The target variable is **quality**, and the remaining columns are input features.

In [46]:
df = pd.read_csv(r"C:\Users\Lenovo\Videos\kaggle\ML Concepts\PCA-Wine-Quality\winequality-red.csv")

# Display first five rows
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [47]:
# Dataset Shape

print("Dataset Shape :", df.shape)

Dataset Shape : (1599, 12)


In [48]:
# Dataset Information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [49]:
# Check Missing Values

df.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

# Standardization of Data

Before applying PCA, it is important to standardize the dataset.

## Why Standardization?

The features in the Wine Quality dataset have different units and scales.

For example:

- Alcohol ranges approximately from **8 to 15**
- Total sulfur dioxide ranges approximately from **6 to 289**

If we apply PCA directly, features with larger values will dominate the covariance matrix.

To avoid this problem, we standardize every feature so that:

- Mean = 0
- Standard Deviation = 1

This ensures that every feature contributes equally to PCA.

In [50]:
X = df.drop("quality", axis=1)

y = df["quality"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (1599, 11)
Target Shape : (1599,)


In [51]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[-0.52835961  0.96187667 -1.39147228 ...  1.28864292 -0.57920652
  -0.96024611]
 [-0.29854743  1.96744245 -1.39147228 ... -0.7199333   0.1289504
  -0.58477711]
 [-0.29854743  1.29706527 -1.18607043 ... -0.33117661 -0.04808883
  -0.58477711]
 ...
 [-1.1603431  -0.09955388 -0.72391627 ...  0.70550789  0.54204194
   0.54162988]
 [-1.39015528  0.65462046 -0.77526673 ...  1.6773996   0.30598963
  -0.20930812]
 [-1.33270223 -1.21684919  1.02199944 ...  0.51112954  0.01092425
   0.54162988]]


## What happened after Standardization?

The original dataset has different scales.

Example:

| Feature | Original Values |
|----------|-----------------|
| Alcohol | 8 - 15 |
| Sulfur Dioxide | 6 - 289 |

After StandardScaler:

- Mean = 0
- Standard Deviation = 1

This makes PCA fair because no feature dominates due to its scale.

In [52]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.528360,0.961877,-1.391472,-0.453218,-0.243707,-0.466193,-0.379133,0.558274,1.288643,-0.579207,-0.960246
1,-0.298547,1.967442,-1.391472,0.043416,0.223875,0.872638,0.624363,0.028261,-0.719933,0.128950,-0.584777
2,-0.298547,1.297065,-1.186070,-0.169427,0.096353,-0.083669,0.229047,0.134264,-0.331177,-0.048089,-0.584777
3,1.654856,-1.384443,1.484154,-0.453218,-0.264960,0.107592,0.411500,0.664277,-0.979104,-0.461180,-0.584777
4,-0.528360,0.961877,-1.391472,-0.453218,-0.243707,-0.466193,-0.379133,0.558274,1.288643,-0.579207,-0.960246


# Mean Calculation

The first mathematical step of PCA is to calculate the mean of every feature.

Formula:

Mean = (x₁ + x₂ + x₃ + ... + xₙ) / n

Since the data has already been standardized, every feature should have a mean close to **0**.

In [53]:

mean = np.mean(X_scaled, axis=0)

print(mean)

fixed acidity           3.554936e-16
volatile acidity        1.733031e-16
citric acid            -8.887339e-17
residual sugar         -1.244227e-16
chlorides               2.132961e-16
free sulfur dioxide    -6.221137e-17
total sulfur dioxide    4.443669e-17
density                -3.473172e-14
pH                      2.861723e-15
sulphates               6.754377e-16
alcohol                 1.066481e-16
dtype: float64


PCA measures the spread of the data around its center.

Therefore, we first find the center (mean) and then shift every data point relative to that center.

This process is called **Mean Centering**.

In [54]:

X_centered = X_scaled - mean
X_centered.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.528360,0.961877,-1.391472,-0.453218,-0.243707,-0.466193,-0.379133,0.558274,1.288643,-0.579207,-0.960246
1,-0.298547,1.967442,-1.391472,0.043416,0.223875,0.872638,0.624363,0.028261,-0.719933,0.128950,-0.584777
2,-0.298547,1.297065,-1.186070,-0.169427,0.096353,-0.083669,0.229047,0.134264,-0.331177,-0.048089,-0.584777
3,1.654856,-1.384443,1.484154,-0.453218,-0.264960,0.107592,0.411500,0.664277,-0.979104,-0.461180,-0.584777
4,-0.528360,0.961877,-1.391472,-0.453218,-0.243707,-0.466193,-0.379133,0.558274,1.288643,-0.579207,-0.960246



Formula:

Centered Data = Original Data − Mean

The purpose of mean centering is to move the center of the data to the origin (0).

After mean centering:

- Positive values indicate observations above the mean.
- Negative values indicate observations below the mean.

This is a required step before computing the covariance matrix.

In [55]:
print(np.mean(X_centered, axis=0))

fixed acidity          -3.554936e-17
volatile acidity       -8.887339e-18
citric acid            -1.777468e-17
residual sugar         -8.887339e-18
chlorides              -8.887339e-18
free sulfur dioxide     0.000000e+00
total sulfur dioxide    2.666202e-17
density                 0.000000e+00
pH                     -1.777468e-17
sulphates               8.887339e-18
alcohol                 0.000000e+00
dtype: float64


# Covariance Matrix

After mean centering, the next step in PCA is to calculate the covariance matrix.

## What is Covariance?

Covariance measures how two features change together.

- Positive Covariance → Both features increase or decrease together.
- Negative Covariance → One feature increases while the other decreases.
- Zero Covariance → No linear relationship between the features.

The covariance matrix contains:

- Variance on the diagonal
- Covariance on the off-diagonal

For a dataset with 11 features, the covariance matrix will have a shape of **11 × 11**.

In [56]:
cov_matrix = np.cov(X_centered.T)

print("Shape :", cov_matrix.shape)
cov_matrix

Shape : (11, 11)


array([[ 1.00062578, -0.25629118,  0.67212377,  0.11484855,  0.09376383,
        -0.15389043, -0.11325227,  0.66846534, -0.68340559,  0.18312019,
        -0.06170686],
       [-0.25629118,  1.00062578, -0.55284143,  0.00191908,  0.06133613,
        -0.0105104 ,  0.07651786,  0.02204002,  0.23508431, -0.26115001,
        -0.20241462],
       [ 0.67212377, -0.55284143,  1.00062578,  0.14366701,  0.20395046,
        -0.06101629,  0.03555526,  0.36517555, -0.54224326,  0.31296577,
         0.10997202],
       [ 0.11484855,  0.00191908,  0.14366701,  1.00062578,  0.05564433,
         0.18716605,  0.20315493,  0.3555057 , -0.08570602,  0.00553058,
         0.04210177],
       [ 0.09376383,  0.06133613,  0.20395046,  0.05564433,  1.00062578,
         0.00556563,  0.04743013,  0.20075788, -0.26519198,  0.37149281,
        -0.22127893],
       [-0.15389043, -0.0105104 , -0.06101629,  0.18716605,  0.00556563,
         1.00062578,  0.66808426, -0.02195956,  0.07042154,  0.0516899 ,
        -0.069

## Understanding the Covariance Matrix

Suppose we have only three features.

| | X | Y | Z |
|---|---|---|---|
| X | Var(X) | Cov(X,Y) | Cov(X,Z) |
| Y | Cov(Y,X) | Var(Y) | Cov(Y,Z) |
| Z | Cov(Z,X) | Cov(Z,Y) | Var(Z) |

### Diagonal Elements

The diagonal values represent the **variance** of each feature.

Example:

Var(X)

Var(Y)

Var(Z)

These values tell us how much each feature varies from its mean.

### Off-Diagonal Elements

These represent covariance.

Example

Cov(X,Y)

If it is

Positive → X and Y increase together.

Negative → One increases while the other decreases.

Zero → No linear relationship.

# Eigenvalues and Eigenvectors

After computing the covariance matrix, PCA calculates its Eigenvalues and Eigenvectors.

## Eigenvalue

An eigenvalue tells us how much variance (information) is captured by a principal component.

A larger eigenvalue means that principal component contains more information.

## Eigenvector

An eigenvector represents the direction of the principal component.

It tells PCA in which direction the data varies the most.

In [57]:
eigen_values, eigen_vectors = np.linalg.eig(cov_matrix)

print("Eigen Values\n")
print(eigen_values)

print("\nEigen Vector Shape :", eigen_vectors.shape)

Eigen Values

[3.10107182 1.92711489 1.55151379 1.21399175 0.95989238 0.05959558
 0.18144664 0.34485779 0.42322138 0.58415655 0.66002104]

Eigen Vector Shape : (11, 11)


In [58]:
# Sort Eigenvalues
idx = np.argsort(eigen_values)[::-1]
eigen_values = eigen_values[idx]
eigen_vectors = eigen_vectors[:, idx]
print(eigen_values)

[3.10107182 1.92711489 1.55151379 1.21399175 0.95989238 0.66002104
 0.58415655 0.42322138 0.34485779 0.18144664 0.05959558]


In [59]:
explained_variance = eigen_values / np.sum(eigen_values)

print(explained_variance)

[0.28173931 0.1750827  0.1409585  0.11029387 0.08720837 0.05996439
 0.05307193 0.03845061 0.0313311  0.01648483 0.00541439]


Cumulative Explained Variance

It represents the total information retained as more principal components are added.

Example

PC1 → 28%

PC1 + PC2 → 46%

PC1 + PC2 + PC3 → 60%

In [60]:
cumulative_variance = np.cumsum(explained_variance)

print(cumulative_variance)

[0.28173931 0.45682201 0.59778051 0.70807438 0.79528275 0.85524714
 0.90831906 0.94676967 0.97810077 0.99458561 1.        ]


In [61]:
# Select Top 2 Principal Components

principal_components = eigen_vectors[:, :2]

print("Principal Components Shape:", principal_components.shape)

Principal Components Shape: (11, 2)


# Data Projection

The final mathematical step in PCA is projecting the original data onto the new principal component axes.

Formula:

Projected Data = Centered Data × Principal Components

This transforms the original 11-dimensional data into a 2-dimensional dataset.

In [62]:
# Project Data
X_pca = X_centered @ principal_components

print("Original Shape :", X_centered.shape)
print("PCA Shape :", X_pca.shape)

Original Shape : (1599, 11)
PCA Shape : (1599, 2)


# PCA Visualization (3D)

To create a 3D visualization, we first compute the first three principal components.

In [63]:
# Select Top 3 Components

principal_components_3d = eigen_vectors[:, :3]

X_pca_3d = X_centered @ principal_components_3d

In [64]:
df_pca3 = pd.DataFrame(
    X_pca_3d,
    columns=["PC1","PC2","PC3"]
)

df_pca3["quality"] = y

In [65]:
fig = px.scatter_3d(
    df_pca3,
    x="PC1",
    y="PC2",
    z="PC3",
    color="quality",
    title="PCA - 3D Visualization"
)

fig.show()

# Logistic Regression WITHOUT PCA

We first train Logistic Regression using all 11 standardized features.

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

start = time.time()

model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

end = time.time()

prediction = model.predict(X_test)

acc_without = accuracy_score(y_test, prediction)

time_without = end - start

print("Accuracy :", acc_without)
print("Training Time :", time_without)

Accuracy : 0.575
Training Time : 0.046190738677978516


# Logistic Regression WITH PCA

Now we train Logistic Regression using only the two principal components.

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

start = time.time()

model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

end = time.time()

prediction = model.predict(X_test)

acc_with = accuracy_score(y_test, prediction)

time_with = end - start

print("Accuracy :", acc_with)
print("Training Time :", time_with)

Accuracy : 0.5125
Training Time : 0.048043012619018555


# Result Comparison

Now let's compare the model performance before and after PCA.

In [68]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy","Training Time"],
    "Without PCA":[acc_without,time_without],
    "With PCA":[acc_with,time_with]
})

comparison

,Metric,Without PCA,With PCA
0,Accuracy,0.575000,0.512500
1,Training Time,0.046191,0.048043


In [69]:
fig = px.bar(
    comparison,
    x="Metric",
    y=["Without PCA","With PCA"],
    barmode="group",
    title="Performance Comparison"
)

fig.show()

# Conclusion

In this project, PCA was implemented manually using NumPy without using `sklearn.PCA()`.

### Steps Performed

- Loaded the Wine Quality Dataset
- Standardized the features
- Calculated the mean
- Performed mean centering
- Computed the covariance matrix
- Calculated eigenvalues and eigenvectors
- Selected the principal components
- Projected the data into a lower-dimensional space
- Visualized the transformed data using Plotly
- Trained Logistic Regression before and after PCA
- Compared accuracy and training time

### Observations

- PCA reduced the number of features from **11 to 2**.
- Training became faster because fewer features were used.
- Most of the important information was preserved.
- PCA is useful for dimensionality reduction, visualization, and reducing computational cost.

This project demonstrates the mathematical implementation of PCA and its practical application in Machine Learning.